In [20]:
from lago.lago import LinkStream, lago_communities
import csv
import pandas as pd

filename = 'data/syn_net.csv'
df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)
time_links = df.values.tolist()

## Initiate empty temporal network (as a link stream)
my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

# NOTE time links can also be imported from txt files with the read_txt() method

## Display linkstream informations
print(f"The link stream consists of {my_linkstream.nb_edges} temporal edges (or time links) accross {my_linkstream.nb_nodes} nodes and {my_linkstream.network_duration} time steps, of which only {my_linkstream.nb_timesteps} contain activity.")


The link stream consists of 182 temporal edges (or time links) accross 20 nodes and 1088 time steps, of which only 182 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_32934/922273661.py:6: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


In [15]:
df.head(5)

,source,destination,timestamp
0,0,7,9
1,0,2,17
2,0,9,25
3,0,18,28
4,0,14,35


In [22]:
import importlib
from lago.lago import LinkStream, lago_communities

my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

## Compute dynamic communities
dynamic_communities = lago_communities(
    my_linkstream,
    nb_iter=1, # run LAGO 3 times and return best result
    )

# Each dynamic community is represented by a list of (<node>, <time instant>)

print(f"{len(dynamic_communities)} dynamic communities have been found")

import importlib
import lago.l_modularity_function as lmf

importlib.reload(lmf)
longitudinal_modularity = lmf.longitudinal_modularity

## Compute Longitudinal Modularity score
## (the higher the better / maximum is 1)
long_mod_score = longitudinal_modularity(
    my_linkstream, 
    dynamic_communities,
    lex_type="MM",
    omega=0.0
    )

print(f"Longitudinal Modularity score of {long_mod_score} ")

6 dynamic communities have been found
community nb interaction:{4: 44, 0: 96, 2: 36, 3: 88, 1: 32, 5: 54}
defaultdict(<class 'float'>, {5: 224, 14: 209, 12: 193, 17: 387, 13: 260, 15: 181, 3: 63, 16: 357, 7: 79, 11: 182, 19: 353, 8: 198, 9: 33, 1: 9})
defaultdict(<class 'float'>, {10: 242, 18: 163, 1: 13, 6: 246, 12: 370, 0: 199, 17: 286})
defaultdict(<class 'float'>, {2: 238, 6: 379, 15: 76, 11: 146, 19: 138, 7: 5, 0: 393, 18: 389, 12: 153, 10: 1})
defaultdict(<class 'float'>, {8: 234, 3: 27, 13: 314, 9: 267, 7: 62, 14: 127, 11: 182, 4: 267, 17: 102, 5: 283, 16: 76, 19: 93, 0: 6, 1: 211})
defaultdict(<class 'float'>, {18: 159, 9: 87, 10: 59, 7: 191, 1: 236, 4: 96, 6: 172, 12: 193, 0: 20, 2: 58, 14: 1, 3: 1})
defaultdict(<class 'float'>, {11: 80, 14: 182, 15: 161, 3: 28, 9: 24, 5: 91, 4: 48, 16: 7, 2: 7, 18: 150, 7: 108, 13: 112, 19: 5, 8: 7})
community expectations mme from @l_modularity_function.py:{0: 0.08583948526027292, 1: 0.019725407065011028, 2: 0.027696426718034368, 3: 0.075308

In [ ]:
for u in 

In [21]:
from __future__ import annotations

from typing import Dict, Hashable, Iterable, Tuple, Any, Optional, Union
import pandas as pd

Node = Hashable
Time = Hashable


def commu_dict_to_interaction_csv(
    commu_dict: Dict[Any, Iterable[Tuple[Node, Time]]],
    syn_csv_path: str = "syn_net.csv",
    out_csv_path: str = "labeled_syn_net.csv",
    *,
    source_col: Union[str, int] = "source",
    destination_col: Union[str, int] = "destination",
    timestamp_col: Union[str, int] = "timestamp",
    syn_header: Optional[int] = "infer",
    syn_sep: str = ",",
    syn_skip_first_row: bool = False,

    node_dtype: Any = int,
    time_dtype: Any = int,

    unknown_commu_value: Any = -1,

    on_conflict: str = "error",
) -> pd.DataFrame:

    syn_df = pd.read_csv(
        syn_csv_path,
        sep=syn_sep,
        header=syn_header,
        skiprows=1 if syn_skip_first_row else None,
        usecols=[source_col, destination_col, timestamp_col],
    ).copy()

    syn_df[source_col] = syn_df[source_col].astype(node_dtype)
    syn_df[destination_col] = syn_df[destination_col].astype(node_dtype)
    syn_df[timestamp_col] = syn_df[timestamp_col].astype(time_dtype)

    active_pairs = set(zip(syn_df[source_col].tolist(), syn_df[timestamp_col].tolist()))
    active_pairs |= set(zip(syn_df[destination_col].tolist(), syn_df[timestamp_col].tolist()))

    node_time_to_commu: Dict[Tuple[Node, Time], Any] = {}

    def _assign(k: Tuple[Node, Time], commu: Any):
        if k not in node_time_to_commu:
            node_time_to_commu[k] = commu
            return
        if node_time_to_commu[k] == commu:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            node_time_to_commu[k] = commu
            return
        raise ValueError(f"Conflict: (node,time)={k} appears in multiple communities: "
                         f"{node_time_to_commu[k]} vs {commu}")

    for commu, pairs in commu_dict.items():
        for u, t in pairs:
            u = node_dtype(u)
            t = time_dtype(t)
            k = (u, t)
            if k in active_pairs:
                _assign(k, commu)

    def _lookup(u: int, t: int):
        return node_time_to_commu.get((u, t), unknown_commu_value)

    syn_df["source_commu"] = [
        _lookup(int(s), int(t)) for s, t in zip(syn_df[source_col], syn_df[timestamp_col])
    ]
    syn_df["destination_commu"] = [
        _lookup(int(d), int(t)) for d, t in zip(syn_df[destination_col], syn_df[timestamp_col])
    ]

    out_df = syn_df.rename(
        columns={
            source_col: "source",
            destination_col: "destination",
            timestamp_col: "timestamp",
        }
    )[["source", "destination", "timestamp", "source_commu", "destination_commu"]]

    out_df.to_csv(out_csv_path, index=False)
    return out_df

real_communities = commu_dict_to_interaction_csv(
    dynamic_communities,
    syn_csv_path="data/syn_net.csv",
    out_csv_path="lago_community.csv",
    syn_header="infer", 
    syn_skip_first_row=False,
    unknown_commu_value=-1, 
    on_conflict="error",
)

(real_communities.head(10))

,source,destination,timestamp,source_commu,destination_commu
0,0,7,9,0,0
1,0,6,17,0,0
2,0,12,25,0,0
3,0,2,28,0,0
4,1,18,35,0,0
5,1,7,39,0,0
6,1,4,48,0,0
7,1,12,52,0,0
8,2,18,60,0,0
9,2,4,69,0,0
